# py-spatialecotyper - a tour of every public function on MERSCOPE melanoma

## 1. What this package does

**Spatial EcoTyper** (Zhang *et al.*, *Nature* 2026,
[doi:10.1038/s41586-026-10452-4](https://doi.org/10.1038/s41586-026-10452-4))
is an unsupervised framework for discovering **spatial ecotypes (SEs)** -
recurrent multicellular communities - from single-cell spatial
transcriptomics. It tiles the tissue into overlapping spatial neighbourhoods,
builds one *cell-type-specific* metacell profile per (neighbourhood, cell
type) pair, reduces each cell type to its own PC space, turns each of those
into a k-nearest-neighbour similarity network, fuses the networks with
similarity network fusion (SNF), clusters the fused graph with Seurat's
Louvain, and - across samples - integrates the per-sample spatial clusters
with consensus NMF so that only the communities that *recur* are reported.

`py-spatialecotyper` is a pure-Python re-implementation of the R package
`SpatialEcoTyper` 1.0.4, validated function by function against R 4.4.3 /
Seurat 5.4.0 / NMF 0.28.

**Input.** A gene x cell matrix (dense `numpy`/`pandas` or `scipy.sparse`;
genes in **rows**) plus a cell metadata `DataFrame` whose index matches the
matrix columns and which carries at least `X`, `Y` and `CellType` columns.
There is no AnnData requirement - any object exposing `.obs` works for the
plotting layer, and the algorithmic layer takes plain arrays.

**Output.** A per-spatial-neighbourhood `SE` label, a per-single-cell `SE`
label, the fused similarity graph, and - for the multi-sample path - the
conserved SE assignment of every spatial cluster in every sample.

This notebook calls **every entry of `pyspatialecotyper.R_FUNCTION_MAP`** on a
real MERSCOPE melanoma sample. Sections:

1. What this package does
2. Install and import
3. Load the demo data
4. One subsection per public function
5. The class API mirror
6. Common pitfalls
7. Where to go next

## 2. Install and import

```bash
pip install pyspatialecotyper
```

The package depends on `numpy`, `scipy`, `pandas`, `scikit-learn`,
`matplotlib` and (optionally) `nmf-rs`; a pure-NumPy fallback for the NMF
kernels is built in.

In [ ]:
import os
import sys
import time

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt

REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO not in sys.path:
    sys.path.insert(0, REPO)

import pyspatialecotyper as se
from pyspatialecotyper import rrandom
from pyspatialecotyper import stats as se_stats
from pyspatialecotyper import nmf as se_nmf
from pyspatialecotyper import network as se_net

print("pyspatialecotyper", se.__version__)
print("public names in __all__ :", len(se.__all__))
print("R_FUNCTION_MAP entries  :", len(se.R_FUNCTION_MAP))

%matplotlib inline
plt.rcParams["figure.dpi"] = 110

`R_FUNCTION_MAP` is the machine-readable R-name to Python-name dictionary.
Every entry below gets its own subsection in section 4.

In [ ]:
rmap = pd.DataFrame({"R function": list(se.R_FUNCTION_MAP),
                     "Python": list(se.R_FUNCTION_MAP.values())})
print(f"{len(rmap)} mapped functions")
rmap

## 3. Load the demo data

The fixture is the Vizgen MERSCOPE human melanoma patient 1 subset used by
upstream Tutorial 1: 500 genes x 27,907 cells, 9 non-malignant cell types,
`Tumor` / `Stroma` region labels.

To keep this notebook fast we take a **deterministic spatial crop** - the
cells nearest the tissue centroid - for the pipeline steps. No RNG is
involved, so the crop is byte-reproducible. `README.md` reports the full
27,907-cell `SpatialEcoTyper` run at **84.1 s** in Python versus 251.8 s in R
on 17 cores; the crop below runs in seconds.

In [ ]:
DATA = os.path.join(REPO, "data")
counts_full = pd.read_csv(os.path.join(DATA, "Melanoma1_subset_counts.tsv.gz"),
                          sep="\t", index_col=0)
meta_full = pd.read_csv(os.path.join(DATA, "Melanoma1_subset_scmeta.tsv"),
                        sep="\t", index_col=0).loc[counts_full.columns]
print("full fixture :", counts_full.shape, "(genes x cells)")
print("metadata cols:", list(meta_full.columns))
meta_full.head()

In [ ]:
N_CROP = 4000
d2 = ((meta_full["X"] - meta_full["X"].median()) ** 2
      + (meta_full["Y"] - meta_full["Y"].median()) ** 2).to_numpy()
crop = np.sort(np.argsort(d2, kind="stable")[:N_CROP])
counts = counts_full.iloc[:, crop]
meta = meta_full.iloc[crop]
genes = list(counts.index)
cells = list(counts.columns)
print("crop         :", counts.shape, "(genes x cells)")
print("cell types   :")
print(meta["CellType"].value_counts())
print("regions      :", meta["Region"].value_counts().to_dict())

Spatial layout of the crop, coloured by cell type. Every downstream section
operates on exactly these cells.

In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 5))
for ct, sub in meta.groupby("CellType"):
    ax.scatter(sub["X"], sub["Y"], s=2, label=ct)
ax.set_xlabel("X"); ax.set_ylabel("Y")
ax.set_title(f"Melanoma1 crop: {len(meta)} cells, {meta['CellType'].nunique()} cell types")
ax.legend(markerscale=4, fontsize=7, loc="upper left", bbox_to_anchor=(1.01, 1.0))
ax.set_aspect("equal")
plt.show()

## 4. One subsection per public function

### 4.A Preprocessing and small utilities

### 4.A.1 `Znorm` -> `znorm`

Z-score normalises each gene across cells. With `groups` it becomes a
**weighted** univariance normalisation in which every group contributes
equally (each cell gets weight `1 / n_group`), so a large region cannot
dominate the gene mean.

```r
znorm_data <- Znorm(normdata)
wtd        <- Znorm(normdata, groups = scmeta$Region)
```

In [ ]:
normdata = se.normalize_data(sp.csc_matrix(counts.to_numpy(float)))
print("normalize_data ->", type(normdata).__name__, normdata.shape)

z_plain = se.znorm(normdata[:40])
z_wtd = se.znorm(normdata[:40], groups=meta["Region"].to_numpy())
print("znorm (no groups) ->", type(z_plain).__name__, z_plain.shape)
print("znorm (weighted)  ->", type(z_wtd).__name__, z_wtd.shape)

fig, ax = plt.subplots(figsize=(5, 3.2))
ax.hist(z_plain.ravel(), bins=80, alpha=0.6, label="Znorm(mat)")
ax.hist(z_wtd.ravel(), bins=80, alpha=0.6, label="Znorm(mat, groups=Region)")
ax.set_yscale("log"); ax.set_xlabel("z-score"); ax.set_ylabel("count (log)")
ax.set_title("Znorm: plain vs region-weighted")
ax.legend()
plt.show()

### 4.A.2 `PreprocessST` -> `preprocess_st`

Drops genes detected in fewer than `min_cells` cells, **then** cells with
fewer than `min_features` detected genes - in that order, so the cell filter
sees the already gene-filtered matrix. Returns the filtered matrix, the
aligned metadata, and the surviving gene / cell names (R returns a list of
two because its matrices carry their own dimnames).

```r
processed <- PreprocessST(normdata, scmeta, min.cells = 5, min.features = 10,
                          X = "X", Y = "Y")
```

In [ ]:
expdat, metadata, genes_kept, cells_kept = se.preprocess_st(
    normdata, meta, min_cells=5, min_features=10, X="X", Y="Y",
    genes=genes, cells=cells)
print("returns a", type((expdat, metadata, genes_kept, cells_kept)).__name__, "of 4")
print("expdat  ->", type(expdat).__name__, expdat.shape)
print("metadata->", type(metadata).__name__, metadata.shape)
print(f"genes {len(genes)} -> {len(genes_kept)}; cells {len(cells)} -> {len(cells_kept)}")

### 4.A.3 `mostFrequent` -> `most_frequent`

Returns the most frequent value of a vector as a string. R builds a `table()`
(levels in **sorted** order) and stably sorts by descending frequency, so a
tie is broken by the lexicographically smallest level - not by first
appearance.

```r
mostFrequent(scmeta$CellType)
```

In [ ]:
print("most_frequent(CellType) ->", repr(se.most_frequent(metadata["CellType"])))
print("most_frequent(Region)   ->", repr(se.most_frequent(metadata["Region"])))
print("tie b,b,a,a,c           ->", repr(se.most_frequent(["b", "b", "a", "a", "c"])),
      "  (smallest level wins the tie)")

### 4.A.4 `rankSparse` -> `rank_sparse`

Replaces the **non-zero** entries of each column by their within-column rank,
divided by `nrow`. Structural zeros are untouched, so the output is a ranking
of each column's support, not of the whole column.

```r
ranked <- rankSparse(fused)
```

In [ ]:
demo = sp.csc_matrix(np.array([[0., 3., 5.],
                               [2., 0., 5.],
                               [7., 3., 1.],
                               [0., 9., 0.]]))
ranked_demo = se.rank_sparse(demo)
print("rank_sparse ->", type(ranked_demo).__name__, ranked_demo.shape)
print("input\n", demo.toarray())
print("output (ties averaged, divided by nrow = 4)\n", ranked_demo.toarray().round(4))

### 4.A.5 `matrixMultiply` -> `matrix_multiply`

Column-blocked `mat1 @ mat2`. The blocking is a memory optimisation only: the
blocks partition the *columns* of `mat2`, never the summation index, so the
product is bit-identical to a single unblocked multiply.

```r
prod <- matrixMultiply(mat1, mat2, minibatch = 100, ncores = 1)
```

In [ ]:
rng_demo = np.arange(1, 31).reshape(3, 10) / 7.0
rhs = np.arange(1, 41).reshape(10, 4) / 3.0
blocked = se.matrix_multiply(rng_demo, rhs, minibatch=2, ncores=1)
print("matrix_multiply ->", type(blocked).__name__, blocked.shape)
print("max abs difference vs a single unblocked product:",
      float(np.max(np.abs(blocked - rng_demo @ rhs))))

### 4.A.6 `getColors` -> `get_colors`

The palette helper behind every plot in the package: 7 categorical palettes
(`kelly`, `cols25`, `polychrome`, `glasbey`, `alphabet2`, `alphabet`, and an
all-colours fallback) and 14 continuous ones. When more colours are requested
than the palette holds, extras are drawn - with replacement, under a
palette-specific seed - from the union of all palettes, which is why the port
needs R's own Mersenne-Twister to reproduce them.

```r
getColors(9, palette = 1, categoric = TRUE)
getColors(64, palette = 1, categoric = FALSE)
```

In [ ]:
cat_pal = se.get_colors(9, palette=1, categoric=True)
cont_pal = se.get_colors(64, palette=1, categoric=False)
print("get_colors ->", type(cat_pal).__name__, "of", len(cat_pal), "hex strings")
print("categorical palette 1, n=9:", cat_pal)

fig, axes = plt.subplots(2, 1, figsize=(7, 2.0))
for ax, pal, title in ((axes[0], cat_pal, "getColors(9, palette=1, categoric=TRUE)"),
                       (axes[1], cont_pal, "getColors(64, palette=1, categoric=FALSE)")):
    for i, c in enumerate(pal):
        ax.add_patch(plt.Rectangle((i, 0), 1, 1, color=c))
    ax.set_xlim(0, len(pal)); ax.set_ylim(0, 1)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title, fontsize=8)
plt.tight_layout()
plt.show()

### 4.B Single-sample spatial ecotype discovery

### 4.B.1 `GetKnnWeights` -> `get_knn_weights`

Builds the cells x neighbourhoods weight matrix behind the metacells: the `k`
nearest cells of one cell type to each neighbourhood centre, masked to those
within `radius`, then **unweighted** (every surviving entry set to 1) and
column-normalised. Returns `(weights, kept_spot_names)`, or `(None, None)`
where R returns `NULL`.

```r
w <- SpatialEcoTyper:::GetKnnWeights(scmeta = tmpmeta, spotCoord = ncmeta,
                                     k = 20, radius = 50)
```

In [ ]:
GRID = round(50 * 1.4)
ncmeta = se.core._spot_metadata(metadata, GRID)
print("spatial neighbourhood grid:", ncmeta.shape, "->", list(ncmeta.columns))

fib = metadata[metadata["CellType"] == "Fibroblast"]
w, kept = se.get_knn_weights(fib, ncmeta, k=20, radius=50, X="X", Y="Y",
                             min_cells_per_region=1)
print("get_knn_weights ->", type(w).__name__, w.shape,
      "| kept spots:", len(kept))
print("column sums (should all be 1):",
      float(np.asarray(w.sum(axis=0)).min()), "..",
      float(np.asarray(w.sum(axis=0)).max()))

### 4.B.2 `GetSpatialMetacells` -> `get_spatial_metacells`

Averages the expression of the `k` nearest cells of each cell type around
each spatial neighbourhood centre. The result is one column per
(neighbourhood, cell type) pair, named `"<SpotID>..<CellType>"` - the `..`
separator every downstream step splits on.

```r
ncem <- GetSpatialMetacells(normdata, metadata, spotCoord = ncmeta,
                            k = 20, radius = 50, ncores = 4)
```

In [ ]:
logexp = expdat.copy()
if logexp.data.max() > 50:
    logexp.data = np.log1p(logexp.data)

ncem, ncem_cols = se.get_spatial_metacells(
    logexp, metadata, X="X", Y="Y", CellType="CellType", spotCoord=ncmeta,
    k=20, radius=50, ncores=1, gene_names=genes_kept, verbose=False)
print("get_spatial_metacells ->", type(ncem).__name__, ncem.shape,
      "| colnames:", type(ncem_cols).__name__, len(ncem_cols))
print("first 3 column names:", ncem_cols[:3])
mc_ct = pd.Series([c.rsplit("..", 1)[1] for c in ncem_cols]).value_counts()
fig, ax = plt.subplots(figsize=(5, 3))
mc_ct.sort_index().plot.bar(ax=ax)
ax.set_ylabel("metacell columns"); ax.set_xlabel("cell type")
ax.set_title("Spatial metacells per cell type")
plt.tight_layout(); plt.show()

### 4.B.3 `GetPCList` -> `get_pc_list`

One PCA per cell type over that cell type's metacell profiles. Returns
`{cell type: (PCs x neighbourhoods embedding, neighbourhood names)}`.

```r
emb_list <- SpatialEcoTyper:::GetPCList(ncem, nfeatures = 300, min.cells = 5,
                                        min.features = 10, ncores = 4)
```

In [ ]:
emb_list = se.get_pc_list(ncem, ncem_cols, genes_kept, nfeatures=300,
                          min_cells=5, min_features=10, ncores=1, verbose=False)
print("get_pc_list ->", type(emb_list).__name__, "with", len(emb_list), "cell types")
for ct, (emb, names) in list(emb_list.items())[:3]:
    print(f"  {ct:12s} embedding {emb.shape} over {len(names)} neighbourhoods")

fig, ax = plt.subplots(figsize=(5, 4))
ct0 = list(emb_list)[0]
emb0, _ = emb_list[ct0]
ax.scatter(emb0[0], emb0[1], s=6)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.set_title(f"GetPCList embedding: {ct0}")
plt.show()

### 4.B.4 `getSN` -> `get_sn`

Turns one cell type's PC embedding into a directed k-NN similarity graph over
the neighbourhoods, with distance mapped to similarity as `1 / (d + 2)` (R
writes it in two steps, so a self-match at distance 0 gets 0.5, not 1).

```r
W <- SpatialEcoTyper:::getSN(emb_list[[1]], k = 50)
```

In [ ]:
sn0 = se.get_sn(emb0[:20], k=50)
print("get_sn ->", type(sn0).__name__, sn0.shape,
      "| stored entries:", sn0.nnz,
      "| value range: %.4f .. %.4f" % (sn0.data.min(), sn0.data.max()))

### 4.B.5 `GetSNList` -> `get_sn_list`

`getSN` for every cell type, after trimming each embedding to `npcs` PCs and
dropping neighbourhoods seen in fewer than `min_cts_per_region` cell types.
Returns `({cell type: (graph, spots)}, spots)` with every graph padded to the
union spot set.

```r
snlist <- SpatialEcoTyper:::GetSNList(emb_list, npcs = 20, k = 50,
                                      min.cts.per.region = 2, ncores = 4)
```

In [ ]:
snlist, sn_spots = se.get_sn_list(emb_list, npcs=20, k=50,
                                  min_cts_per_region=2, ncores=1, verbose=True)
print("get_sn_list ->", type(snlist).__name__, "of", len(snlist), "graphs;",
      len(sn_spots), "shared neighbourhoods")
nnz = pd.Series({ct: w.nnz for ct, (w, _) in snlist.items()}).sort_index()
fig, ax = plt.subplots(figsize=(5, 3))
nnz.plot.bar(ax=ax)
ax.set_ylabel("stored similarities"); ax.set_title("GetSNList: graph density per cell type")
plt.tight_layout(); plt.show()

### 4.B.6 `fillspots` -> `fillspots`

Pads every network in a list to the **union** of the spot sets, in
first-appearance order, so the fusion can add them elementwise. Missing
neighbourhoods become all-zero rows and columns.

```r
snlist <- SpatialEcoTyper:::fillspots(snlist)
```

In [ ]:
raw = {ct: (se.get_sn(emb[:20], k=50), names) for ct, (emb, names) in
       list(emb_list.items())[:3]}
print("before fillspots, spot counts:", {ct: len(n) for ct, (_, n) in raw.items()})
filled, union_spots = se.fillspots(raw)
print("fillspots ->", type(filled).__name__, "| union spot set:", len(union_spots))
print("after  fillspots, shapes    :", {ct: w.shape for ct, (w, _) in filled.items()})

### 4.B.7 `.dominateset` -> `dominateset`

Sparsifies a similarity matrix by keeping only the `KK` largest entries of
each row. Two R edge cases are reproduced verbatim: at `n == KK` R's `1:0`
quirk zeroes the single smallest element, and at `n < KK` R aborts.

```r
SpatialEcoTyper:::.dominateset(W, KK = 10, ncores = 1)
```

In [ ]:
dom = se.dominateset(sn0, KK=10, ncores=1)
print("dominateset ->", type(dom).__name__, dom.shape,
      "| nnz %d -> %d" % (sn0.nnz, dom.nnz),
      "| max entries per row:", int(np.diff(sp.csr_matrix(dom).indptr).max()))

### 4.B.8 `SNF2` -> `snf2`

Enhanced similarity network fusion. Each view is row-normalised, reduced to
its `K` dominant neighbours, and cross-diffused for `t` rounds. The final
average is taken **only over the views that actually observed each pair**,
which is upstream's modification that lets a neighbourhood be missing from
some cell types without dragging its fused similarity to zero.

```r
fused <- SNF2(snlist, K = 10, t = 10, minibatch = 5000, ncores = 4)
```

In [ ]:
t0 = time.time()
fused = se.snf2([w for w, _ in snlist.values()], K=10, t=10, minibatch=5000,
                ncores=1, verbose=False)
print("snf2 ->", type(fused).__name__, fused.shape,
      "| %.2f s for %d views" % (time.time() - t0, len(snlist)))
ranked = se.rank_sparse(fused)
fig, ax = plt.subplots(figsize=(4.6, 4))
im = ax.imshow(ranked.toarray()[:80, :80], cmap="magma")
ax.set_title("SNF2 fused graph after rankSparse (first 80 x 80)")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.show()

### 4.B.9 `SpatialEcoTyper` -> `spatial_ecotyper`

The single-sample entry point: everything above, plus Seurat's
`ScaleData -> RunPCA -> FindNeighbors -> FindClusters` and `AnnotateCells`.

```r
se_results <- SpatialEcoTyper(normdata, scmeta, outprefix = NULL, radius = 50,
                              resolution = 0.5, nfeatures = 300, ncores = 4)
```

In [ ]:
t0 = time.time()
result = se.spatial_ecotyper(normdata, meta, gene_names=genes, outprefix=None,
                             radius=50, resolution=0.5, nfeatures=300,
                             min_cts_per_region=2, npcs=20, min_cells=5,
                             min_features=10, iterations=10, minibatch=5000,
                             ncores=1, k=20, k_sn=50, dropcell=True,
                             verbose=False)
elapsed_single = time.time() - t0
print("spatial_ecotyper ->", type(result).__name__, "in %.2f s" % elapsed_single)
print("  spot_metadata:", result.spot_metadata.shape)
print("  metadata     :", result.metadata.shape)
print("  fused        :", result.fused.shape)
print("  SEs found    :", sorted(set(result.spot_metadata["SE"])))
result.spot_metadata.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
for ax, (frame, key, title) in zip(
        axes,
        [(result.spot_metadata, "SE", "Spatial neighbourhoods"),
         (result.metadata, "SE", "Single cells")]):
    for lab, sub in frame.groupby(key):
        ax.scatter(sub["X"], sub["Y"], s=6 if title.startswith("Spatial") else 2,
                   label=lab)
    ax.set_title(f"{title} coloured by SE")
    ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_aspect("equal")
axes[1].legend(markerscale=4, fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout(); plt.show()

### 4.B.10 `AnnotateCells` -> `annotate_cells`

Transfers a neighbourhood-level label to single cells by 1-nearest-neighbour,
blanking cells whose nearest neighbourhood is farther than `radius`. R reads
`radius` out of the Seurat object's `project.name` string; here it is an
explicit argument.

```r
scmeta <- AnnotateCells(scmeta = scmeta, obj = se_results$obj,
                        col = "SE", dropcell = TRUE)
```

In [ ]:
ann_drop = se.annotate_cells(metadata, result.spot_metadata, radius=50,
                             col="SE", dropcell=True)
ann_keep = se.annotate_cells(metadata, result.spot_metadata, radius=50,
                             col="SE", dropcell=False)
print("annotate_cells ->", type(ann_drop).__name__, ann_drop.shape,
      "| dropcell=False keeps", ann_keep.shape[0], "rows")
print("cells with no SE (dropcell=False):", int(ann_keep["SE"].isna().sum()))
print(ann_drop["SE"].value_counts().sort_index())

### 4.C Multi-sample integration

The integration layer needs at least two samples, so we first load a second
MERSCOPE subset (colorectal cancer sample 2, upstream Tutorial 2), take the
same deterministic 8,000-cell crop from each, restrict to the genes the two
panels share, and run `spatial_ecotyper` on both.

In [ ]:
def load_crop(tag, n):
    c = pd.read_csv(os.path.join(DATA, f"{tag}_subset_counts.tsv.gz"),
                    sep="\t", index_col=0)
    m = pd.read_csv(os.path.join(DATA, f"{tag}_subset_scmeta.tsv"),
                    sep="\t", index_col=0).loc[c.columns]
    dd = ((m["X"] - m["X"].median()) ** 2 + (m["Y"] - m["Y"].median()) ** 2).to_numpy()
    kk = np.sort(np.argsort(dd, kind="stable")[:n])
    return c.iloc[:, kk], m.iloc[kk]


skcm_counts, skcm_meta = load_crop("Melanoma1", 8000)
crc_counts, crc_meta = load_crop("CRC2", 8000)
shared = [g for g in skcm_counts.index if g in set(crc_counts.index)]
print("shared genes between the two panels:", len(shared))
skcm_counts = skcm_counts.loc[shared]
crc_counts = crc_counts.loc[shared]
skcm_norm = se.normalize_data(sp.csc_matrix(skcm_counts.to_numpy(float)))
crc_norm = se.normalize_data(sp.csc_matrix(crc_counts.to_numpy(float)))

res_skcm = se.spatial_ecotyper(skcm_norm, skcm_meta, gene_names=shared,
                               radius=50, min_cts_per_region=1, ncores=1,
                               verbose=False)
res_crc = se.spatial_ecotyper(crc_norm, crc_meta, gene_names=shared,
                              radius=50, min_cts_per_region=1, ncores=1,
                              verbose=False)
print("SKCM:", res_skcm, "\nCRC :", res_crc)

### 4.C.1 `ComputeFCs` -> `compute_fcs`

Per cell type, the log fold change of every gene in each spatial cluster
versus **all other cells of that cell type** (not versus all cells). Row
names become `"<CellType>..<gene>"`, the key the integration step splits on.

```r
fcs <- SpatialEcoTyper:::ComputeFCs(normdata, scmeta, cluster = "SE",
                                    scale = TRUE, ncores = 1)
```

In [ ]:
def sample_fcs(res, norm_mat, all_cells, gene_list, tag, subresolution=30):
    """Reproduce what `integrate_spatial_ecotyper` does before `Integrate`:
    over-cluster each sample's SNN graph at `subresolution` into `InitSE`
    spatial clusters, transfer them to cells, then take fold changes."""
    cl = se.find_clusters(res.snn, resolution=subresolution, random_seed=0)
    spot_meta = res.spot_metadata.copy()
    spot_meta["SE"] = [f"{tag}..InitSE{c}" for c in cl]
    md = se.annotate_cells(res.metadata.drop(columns=["SE"]), spot_meta,
                           radius=50, col="SE", dropcell=True)
    idx = np.array([all_cells.index(c) for c in md.index])
    return se.compute_fcs(norm_mat[:, idx], md, gene_list, cluster="SE",
                          scale=True, ncores=1)


fc, fc_rows, fc_ses = sample_fcs(res_skcm, skcm_norm, list(skcm_counts.columns),
                                 shared, "SKCM")
fc2, fc2_rows, fc2_ses = sample_fcs(res_crc, crc_norm, list(crc_counts.columns),
                                    shared, "CRC")
print("compute_fcs -> tuple(matrix, rownames, colnames)")
print("  SKCM matrix:", type(fc).__name__, fc.shape, "over", len(fc_ses), "clusters")
print("  CRC  matrix:", type(fc2).__name__, fc2.shape, "over", len(fc2_ses), "clusters")
print("  rownames   :", fc_rows[:2], "...")
print("  colnames   :", fc_ses[:3], "...")

### 4.C.2 `Integrate` -> `integrate`

Builds one rank-normalised correlation network per cell type across every
sample's spatial clusters and fuses them with `snf2`. The rank normalisation
is what makes samples comparable: within each source sample every column of
the correlation sub-matrix becomes a probability distribution over that
sample's clusters.

```r
integ <- SpatialEcoTyper:::Integrate(avgexprs, nfeatures = 200,
                                     min.features = 5, seed = 1)
```

In [ ]:
common_rows = [r for r in fc_rows if r in set(fc2_rows)]
i1 = {r: i for i, r in enumerate(fc_rows)}
i2 = {r: i for i, r in enumerate(fc2_rows)}
avg = np.column_stack([fc[[i1[r] for r in common_rows]],
                       fc2[[i2[r] for r in common_rows]]])
avg_cols = list(fc_ses) + list(fc2_ses)
print("avgexprs:", avg.shape, "=", len(common_rows),
      "shared (celltype..gene) rows x", len(avg_cols), "spatial clusters")
integrated, integ_spots = se.integrate(avg, common_rows, avg_cols,
                                       nfeatures=200, min_features=5,
                                       minibatch=5000, ncores=1, seed=1,
                                       verbose=False)
print("integrate -> tuple(matrix, spots)")
print("  matrix:", type(integrated).__name__, integrated.shape)
print("  spots :", integ_spots)
fig, ax = plt.subplots(figsize=(4.4, 3.8))
im = ax.imshow(np.asarray(integrated.todense()), cmap="viridis")
ax.set_xticks(range(len(integ_spots))); ax.set_yticks(range(len(integ_spots)))
ax.set_xticklabels(integ_spots, rotation=90, fontsize=6)
ax.set_yticklabels(integ_spots, fontsize=6)
ax.set_title("Integrate: cross-sample cluster similarity")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.show()

### 4.C.3 `nmfClustering` -> `nmf_clustering`

Consensus NMF over `nrun_per_rank` random restarts per rank. The restart
seeds are `sample(1:6280, nrun.per.rank)` after `set.seed(seed)`, reproduced
exactly through `pyspatialecotyper.rrandom`. `bestK` is the largest rank whose
cophenetic coefficient exceeds `min_coph`, else the argmax.

```r
fit <- nmfClustering(mat, ranks = 2:5, nrun.per.rank = 10, seed = 2024)
```

In [ ]:
dense_int = np.asarray(integrated.todense())
clust = se.nmf_clustering(dense_int, row_names=integ_spots, col_names=integ_spots,
                          ranks=range(2, 6), nrun_per_rank=10, seed=2024,
                          ncores=1, verbose=False)
print("nmf_clustering ->", type(clust).__name__, "with keys", list(clust))
print("bestK :", clust["bestK"])
print("labels:", clust["labels"])
fig, ax = plt.subplots(figsize=(4.6, 3))
ax.plot(clust["cophenetic"]["K"], clust["cophenetic"]["Cophenetic"], "o-")
ax.axvline(clust["bestK"], color="black", ls="--", label=f"bestK = {clust['bestK']}")
ax.set_xlabel("rank K"); ax.set_ylabel("cophenetic coefficient")
ax.set_title("nmfClustering rank selection"); ax.legend()
plt.tight_layout(); plt.show()

### 4.C.4 `IntegrateSpatialEcoTyper` -> `integrate_spatial_ecotyper`

The whole integration step - `ComputeFCs` per sample, `Integrate`,
`nmfClustering`, then writing the conserved SE back onto the single cells -
starting from `SpatialEcoTyperResult` objects you already have. Use this when
the per-sample runs were done separately (e.g. on a cluster).

```r
IntegrateSpatialEcoTyper(SpatialEcoTyper_list, data_list, outdir = "./",
                         nmf.ranks = 4:6, nrun.per.rank = 5, seed = 1)
```

In [ ]:
t0 = time.time()
integ2 = se.integrate_spatial_ecotyper(
    result_list={"SKCM": res_skcm, "CRC": res_crc},
    data_list={"SKCM": skcm_norm, "CRC": crc_norm},
    gene_names=shared, nmf_ranks=range(4, 7), nrun_per_rank=5,
    ncores=1, seed=1, verbose=False)
print("integrate_spatial_ecotyper -> dict with keys", list(integ2),
      "in %.1f s" % (time.time() - t0))
print("integrated matrix:", integ2["integrated"].shape)
print("metadata         :", integ2["metadata"].shape)
print("conserved SE per sample:")
print(integ2["cluster_SE"].groupby(["Sample", "SE"]).size().unstack(fill_value=0))

### 4.C.5 `MultiSpatialEcoTyper` -> `multi_spatial_ecotyper`

The end-to-end wrapper: `spatial_ecotyper` per sample followed by
`integrate_spatial_ecotyper`, in one call. Given the same inputs and seed it
reproduces section 4.C.4 exactly.

```r
MultiSpatialEcoTyper(data_list, metadata_list, outdir = "./",
                     nmf.ranks = 4:6, nrun.per.rank = 5, seed = 1)
```

In [ ]:
t0 = time.time()
multi = se.multi_spatial_ecotyper(
    data_list={"SKCM": skcm_norm, "CRC": crc_norm},
    metadata_list={"SKCM": skcm_meta, "CRC": crc_meta},
    gene_names=shared, nmf_ranks=range(4, 7), nrun_per_rank=5,
    min_cts_per_region=1, ncores=1, seed=1, verbose=False)
print("multi_spatial_ecotyper -> dict with keys", list(multi),
      "in %.1f s" % (time.time() - t0))
print("same conserved SE assignment as the two-step route above:",
      bool((multi["cluster_SE"]["SE"].to_numpy()
            == integ2["cluster_SE"].loc[multi["cluster_SE"].index, "SE"].to_numpy()).all()))
fig, ax = plt.subplots(figsize=(6.2, 4.6))
mm = multi["metadata"]
for lab, sub in mm[mm["Sample"] == "CRC"].groupby("SE"):
    ax.scatter(sub["X"], sub["Y"], s=2, label=lab)
ax.set_aspect("equal"); ax.set_title("Conserved SEs mapped onto the CRC sample")
ax.legend(markerscale=4, fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout(); plt.show()

### 4.D The NMF recovery layer

### 4.D.1 `NMFGenerateW` -> `nmf_generate_w`

Trains a single SE-deconvolution basis matrix `W` (features x SEs) from a
fraction matrix and an expression matrix. `H` is held fixed at the fractions,
so only the `W` update runs and the KL objective is convex. Features are then
filtered by `delta = W - (second largest value in the row)`.

```r
W <- NMFGenerateW(Fracs, ExpMat, scale = TRUE, nfeature = 300,
                  nfeature.per.se = 50, method = "brunet")
```

In [ ]:
sub_idx = np.array([cells.index(c) for c in result.metadata.index])
sub_norm = np.asarray(normdata[:, sub_idx].todense())
se_lab = result.metadata["SE"].astype(str).to_numpy()
se_levels = sorted(set(se_lab))
Fracs = pd.DataFrame(
    np.eye(len(se_levels))[[se_levels.index(s) for s in se_lab]],
    index=result.metadata.index, columns=se_levels)
W, w_names, w_ses = se.nmf_generate_w(Fracs.to_numpy(), sub_norm,
                                      feature_names=genes,
                                      se_names=se_levels, scale=True,
                                      nfeature=300, nfeature_per_se=50,
                                      method="brunet")
print("nmf_generate_w -> tuple(W, rownames, colnames)")
print("  W        :", type(W).__name__, W.shape)
print("  rownames :", w_names[:3], "...")
print("  colnames :", w_ses)
fig, ax = plt.subplots(figsize=(4.2, 4))
im = ax.imshow(W[np.argsort(-W.max(axis=1))][:60], aspect="auto", cmap="magma")
ax.set_xticks(range(len(w_ses))); ax.set_xticklabels(w_ses, rotation=90)
ax.set_ylabel("feature (top 60 by max loading)")
ax.set_title("NMFGenerateW basis W")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.show()

### 4.D.2 `NMFGenerateWList` -> `nmf_generate_w_list`

One `W` per cell type. Cell types are kept only when they have more than one
SE with more than `min_cells` cells. All `sample()` calls run on R's
Mersenne-Twister, so the balancing and down-sampling pick the cells R would.

```r
Ws <- NMFGenerateWList(scdata, scmeta, CellType = "CellType", SE = "SE",
                       seed = 2024)
```

In [ ]:
Ws = se.nmf_generate_w_list(sub_norm, result.metadata, gene_names=genes,
                            CellType="CellType", SE="SE", scale=True,
                            Sample=None, min_cells=20, downsample=2500,
                            ncores=1, seed=2024, verbose=False)
print("nmf_generate_w_list ->", type(Ws).__name__, "keyed by cell type:", list(Ws))
for ct in list(Ws)[:3]:
    Wc, nm, ss = Ws[ct]
    print(f"  {ct:12s} W {Wc.shape}  over SEs {ss}")

### 4.D.3 `NMFpredict` -> `nmf_predict`

Projects new data onto a pretrained `W`, holding `W` fixed so only `H` is
updated. Cells are processed in contiguous chunks of at most `ncell_per_run`
whose sizes differ by at most 1, and the result is transposed to cells x SE.

```r
preds <- NMFpredict(W = W, testdat = bulkdata, scale = TRUE,
                    ncell.per.run = 500, sum2one = TRUE)
```

In [ ]:
H, H_ses, H_cells = se.nmf_predict(W, w_names, w_ses, sub_norm[:, :400],
                                   genes, list(result.metadata.index[:400]),
                                   scale=False, ncell_per_run=200,
                                   sum2one=True, ncores=1)
print("nmf_predict -> tuple(H, se_names, cell_names)")
print("  H:", type(H).__name__, H.shape, "(cells x SE)")
print("  row sums (sum2one=True):", float(H.sum(axis=1).min()), "..",
      float(H.sum(axis=1).max()))
fig, ax = plt.subplots(figsize=(5.4, 3))
pd.DataFrame(H, columns=H_ses).boxplot(ax=ax)
ax.set_ylabel("NMF score"); ax.set_title("NMFpredict scores per SE")
plt.tight_layout(); plt.show()

### 4.D.4 `.nmf.predict` -> `nmf._nmf_predict`

The internal single-chunk kernel behind `NMFpredict`. It calls `set.seed(39)`
before initialising `H`, so this path is deterministic end to end. It returns
`H` in SE x cells orientation - `NMFpredict` is the one that transposes.

```r
SpatialEcoTyper:::.nmf.predict(W, testdat, scale = FALSE, normalize = TRUE)
```

In [ ]:
h_raw, h_ses = se_nmf._nmf_predict(W, w_names, w_ses, sub_norm[:, :100],
                                   genes, list(result.metadata.index[:100]),
                                   scale=False, normalize=True)
print("_nmf_predict -> tuple(H, se_names)")
print("  H:", type(h_raw).__name__, h_raw.shape, "(SE x cells - note the orientation)")
print("  identical to nmf_predict's transpose on the same 100 cells:",
      bool(np.allclose(h_raw.T, se.nmf_predict(W, w_names, w_ses,
                                               sub_norm[:, :100], genes,
                                               list(result.metadata.index[:100]),
                                               scale=False, ncell_per_run=5000,
                                               sum2one=True)[0])))

### 4.D.5 `RecoverSE` -> `recover_se`

Scores every cell against the per-cell-type models, hard-assigns it to the
argmax SE, and demotes calls below `min_score` to `"NonSE"`. Only the
`Ws = <custom>` branch is ported: upstream's default loads the MERSCOPE
models from the R package's `inst/extdata`, which are not redistributable.

```r
calls <- RecoverSE(dat, celltypes = scmeta$CellType, Ws = Ws, min.score = 0.6)
```

In [ ]:
calls = se.recover_se(sub_norm, genes, list(result.metadata.index),
                      result.metadata["CellType"].to_numpy(), Ws,
                      scale=True, ncell_per_run=500, min_score=0.6,
                      ncores=1, verbose=False)
print("recover_se ->", type(calls).__name__, calls.shape, list(calls.columns))
print(calls["SE"].value_counts().sort_index())
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(calls["PredScore"], bins=50)
ax.axvline(0.6, color="black", ls="--", label="min_score = 0.6")
ax.set_xlabel("PredScore"); ax.set_ylabel("cells")
ax.set_title("RecoverSE confidence"); ax.legend()
plt.tight_layout(); plt.show()

### 4.D.6 `AggregateRecoverModels` -> `aggregate_recover_models`

Averages several per-cell-type `W` matrices, keeping only features selected
as SE-specific in more than `min_model_fraction` of the models. The average
ignores NA, so a feature missing from one model does not drag its mean down.

```r
AggregateRecoverModels(model_list, delta.threshold = 0.01,
                       min.model.fraction = 0.5)
```

In [ ]:
agg = se.aggregate_recover_models({"fold1": Ws, "fold2": Ws},
                                  delta_threshold=0.01, min_model_fraction=0.5)
print("aggregate_recover_models ->", type(agg).__name__, "keyed by", list(agg))
ct0m = list(agg)[0]
print(f"  {ct0m}: W {agg[ct0m][0].shape} over SEs {agg[ct0m][2]}")

### 4.D.7 `LoocvPredict` -> `loocv_predict`

Leave-one-sample-out cross-validation of the recovery models. With a single
sample R instead splits each cell type in half at random. R's version calls
`sample(1:10000, repeats)` *without* setting a seed first, so the port adds an
explicit `seed=` argument to make it reproducible.

```r
LoocvPredict(scdata, scmeta, Sample = "Sample", CellType = "CellType",
             SE = "SE", repeats = 30)
```

In [ ]:
halves = se.partition_tissue(result.metadata, nrow=2, ncol=1, x="X", y="Y")
print("pseudo-samples for the LOOCV split:",
      halves["Partition"].value_counts().to_dict())
loocv = se.loocv_predict(sub_norm, halves, genes, Sample="Partition",
                         CellType="CellType", SE="SE", repeats=1,
                         ncores=1, scale=True, seed=1, verbose=False)
print("loocv_predict ->", type(loocv).__name__, loocv.shape)
loocv.head()

### 4.D.8 `DeconvoluteSE` -> `deconvolute_se`

Bulk deconvolution: pushes a bulk expression matrix through a trained `W` and
returns a samples x SE fraction matrix.

```r
fracs <- DeconvoluteSE(bulkdata, W = W, scale = TRUE, sum2one = TRUE)
```

In [ ]:
rrandom.set_seed(1)
pseudo = se.create_pseudobulks(counts=counts.loc[:, result.metadata.index],
                               groups=result.metadata["SE"], n_mixtures=12)
Wct, wn_ct, ws_ct = Ws[list(Ws)[0]]
fracs, frac_ses, frac_samples = se.deconvolute_se(
    pseudo["Mixtures"].to_numpy(), list(pseudo["Mixtures"].index),
    list(pseudo["Mixtures"].columns), Wct, wn_ct, ws_ct,
    scale=True, nsample_per_run=500, sum2one=True, ncores=1)
print("deconvolute_se -> tuple(fracs, se_names, sample_names)")
print("  fracs:", type(fracs).__name__, fracs.shape, "(pseudobulks x SE)")
fig, ax = plt.subplots(figsize=(5.6, 3))
pd.DataFrame(fracs, index=frac_samples, columns=frac_ses).plot.bar(
    stacked=True, ax=ax)
ax.set_ylabel("estimated fraction"); ax.set_xlabel("pseudobulk")
ax.set_title("DeconvoluteSE on synthetic pseudobulks")
ax.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout(); plt.show()

### 4.D.9 `CreatePseudobulks` -> `create_pseudobulks`

Builds pseudobulk mixtures with random group fractions drawn from R's own
`rnorm` / `sample` stream, returning both the mixture matrix and the ground
truth fractions used to build it.

```r
set.seed(1)
pb <- CreatePseudobulks(counts = scdata, groups = groups, n_mixtures = 12)
```

In [ ]:
print("create_pseudobulks ->", type(pseudo).__name__, "with keys", list(pseudo))
print("  Fracs   :", pseudo["Fracs"].shape, "(pseudobulk x group)")
print("  Mixtures:", pseudo["Mixtures"].shape, "(gene x pseudobulk)")
pseudo["Fracs"].round(3).head()

### 4.D.10 `InferNCells` -> `infer_ncells`

Linearly rescales each spot's total expression so the lowest-expressing spot
gets 1 cell and the mean is `avg_number`. Used for Visium-style data where
the number of cells per spot is unknown.

```r
ncells <- InferNCells(scdata, avg.number = 5)
```

In [ ]:
nc = se.infer_ncells(counts.to_numpy(float), avg_number=5)
print("infer_ncells ->", type(nc).__name__, nc.shape, nc.dtype)
print("  min %d, mean %.3f, max %d" % (nc.min(), nc.mean(), nc.max()))
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(nc, bins=40)
ax.set_xlabel("inferred cells per spot"); ax.set_ylabel("spots")
ax.set_title("InferNCells(avg.number = 5)")
plt.tight_layout(); plt.show()

### 4.E Downstream statistics

For the statistics layer we need per-sample structure. `PartitionTissue`
gives it deterministically.

### 4.E.1 `PartitionTissue` -> `partition_tissue`

Splits the tissue into an `nrow` x `ncol` grid of subregions and adds a
`Partition` column of the form `"<row>_<col>"`. Note the R quirk that `nrow`
divides the **X** axis, which reads backwards but is reproduced verbatim.

```r
SpatialEcoTyper:::PartitionTissue(meta, nrow = 3, ncol = 3, X = "X", Y = "Y")
```

In [ ]:
stat_meta = se.partition_tissue(result.metadata, nrow=3, ncol=3, x="X", y="Y")
stat_meta = stat_meta.rename(columns={"Partition": "Sample"})
print("partition_tissue ->", type(stat_meta).__name__, stat_meta.shape)
print(stat_meta["Sample"].value_counts().sort_index())
fig, ax = plt.subplots(figsize=(5, 4.6))
for lab, sub in stat_meta.groupby("Sample"):
    ax.scatter(sub["X"], sub["Y"], s=3, label=lab)
ax.set_aspect("equal"); ax.set_title("PartitionTissue(nrow=3, ncol=3)")
ax.legend(markerscale=4, fontsize=7, loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout(); plt.show()

### 4.E.2 `Coassociation` -> `coassociation`

Pairwise co-association (Pearson correlation across samples) between cell
states, where a cell state is `"<SE>_<CellType>"`. Abundances are computed
under four inclusion / fill schemes and the four correlation matrices are
averaged ignoring missing values.

```r
coassoc <- Coassociation(scmeta, Sample = "Sample", SE = "SE",
                         CellType = "CellType", NonSE = "NonSE", test = FALSE)
```

In [ ]:
coassoc = se.coassociation(stat_meta, sample="Sample", se="SE",
                           cell_type="CellType", non_se="NonSE", test=False)
print("coassociation ->", type(coassoc).__name__, coassoc.shape)
fig, ax = plt.subplots(figsize=(5.2, 4.6))
im = ax.imshow(coassoc.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(coassoc.shape[1])); ax.set_yticks(range(coassoc.shape[0]))
ax.set_xticklabels(coassoc.columns, rotation=90, fontsize=5)
ax.set_yticklabels(coassoc.index, fontsize=5)
ax.set_title("Coassociation index (cell state x cell state)")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.show()

### 4.E.3 `CoassociationTest` -> `coassociation_test`

Permutation significance of within-SE co-association: the mean within-SE
score is compared against a null built from `nperm` full column permutations
of the matrix. The signed Z-scores come back in `result.attrs["Zscore"]`,
which is R's `attr(Pvals, "Zscore")`.

```r
set.seed(1)
pvals <- CoassociationTest(coassoc, nperm = 1000)
attr(pvals, "Zscore")
```

In [ ]:
rrandom.set_seed(1)
cpv = se.coassociation_test(coassoc, nperm=1000)
print("coassociation_test ->", type(cpv).__name__, cpv.shape)
print(pd.DataFrame({"P": cpv, "Z": pd.Series(cpv.attrs["Zscore"])}))

### 4.E.4 `.colocalization` -> `stats._colocalization`

The observed-value kernel behind `Colocalization`: for every cell, the
composition of its within-radius neighbourhood over cell states, aggregated
into a state x state matrix. No permutation, no RNG.

```r
SpatialEcoTyper:::.colocalization(scmeta, coords = c("X", "Y"),
                                  CellState = "CellState", radius = 50,
                                  k = 100, min.cell = 10)
```

In [ ]:
cs_meta = stat_meta.copy()
cs_meta["CellState"] = (cs_meta["SE"].astype(str) + "_"
                        + cs_meta["CellType"].astype(str))
obs = se_stats._colocalization(cs_meta, coords=("X", "Y"),
                               cell_state="CellState", radius=50, k=100,
                               min_cell=10)
print("_colocalization ->", type(obs).__name__, obs.shape)
print(obs.iloc[:4, :4].round(4))

### 4.E.5 `Colocalization` -> `colocalization`

Z-scores the observed colocalization against a null built by shuffling cell
states **within** each cell type, preserving cell-type composition. `ncores`
is accepted for signature parity only: R's `mclapply` forks and each child
reseeds from its PID, so `ncores > 1` is not reproducible in R either - the
port matches the `ncores = 1` branch.

```r
set.seed(1)
coloc <- Colocalization(scmeta, coords = c("X", "Y"), SE = "SE",
                        CellType = "CellType", radius = 50, k = 100,
                        min.cell = 10, nperm = 200, ncores = 1)
```

In [ ]:
rrandom.set_seed(1)
coloc = se.colocalization(stat_meta, coords=("X", "Y"), se="SE",
                          cell_type="CellType", radius=50, k=100,
                          min_cell=10, nperm=200, test=True, ncores=1)
print("colocalization ->", type(coloc).__name__, "with keys", list(coloc))
print("  ColocIndex:", coloc["ColocIndex"].shape)
print("  Pval      :", coloc["Pval"].to_dict())

### 4.E.6 `ColocalizationMetaAnalysis` -> `colocalization_meta_analysis`

Stouffer-combines per-sample colocalization results, clipping each sample's
index at `+/- cap` and keeping only states seen in at least `min_samples`
samples.

```r
ColocalizationMetaAnalysis(coloc_list, cap = 5, min.samples = 1)
```

In [ ]:
thirds = pd.qcut(stat_meta["X"], 3, labels=["T1", "T2", "T3"])
coloc_list = []
for i, lab in enumerate(["T1", "T2", "T3"]):
    rrandom.set_seed(100 + i)
    coloc_list.append(se.colocalization(stat_meta[thirds == lab],
                                        coords=("X", "Y"), se="SE",
                                        cell_type="CellType", radius=50, k=100,
                                        min_cell=10, nperm=50, test=True,
                                        ncores=1))
meta_coloc = se.colocalization_meta_analysis(coloc_list, cap=5, min_samples=1)
print("colocalization_meta_analysis ->", type(meta_coloc).__name__,
      "with keys", list(meta_coloc))
print("  MetaColocIndex:", meta_coloc["MetaColocIndex"].shape)
print("  MetaPval      :", meta_coloc["MetaPval"].to_dict())

### 4.E.7 `.moran` -> `stats._moran`

Moran's I of each SE's binary indicator, in closed form:
`I = (n / S0) * (z' W z) / sum(z^2)`. Under `style = "W"` every row of `W`
sums to 1, so `S0 = n` and the leading factor is 1.

```r
SpatialEcoTyper:::.moran(scmeta$SE, listw, ncores = 1)
```

In [ ]:
xy = stat_meta[["X", "Y"]].to_numpy(float)
knn_idx, _ = se_stats._knn(xy, xy, 4)
rows = np.repeat(np.arange(len(xy)), 3)
listw = sp.csr_matrix((np.full(rows.size, 1 / 3), (rows, knn_idx[:, 1:].ravel())),
                      shape=(len(xy), len(xy)))
moran_obs = se_stats._moran(stat_meta["SE"].to_numpy(), listw, ncores=1)
print("_moran ->", type(moran_obs).__name__)
print(moran_obs)

### 4.E.8 `ComputeNormalizedMoranI` -> `compute_normalized_moran_i`

Normalises Moran's I by a permutation null in which SE labels are shuffled
*within* each cell type, returning a Z-score per SE. The R reference for this
function is produced with `spdep`'s `knearneigh` / `nb2listw`.

```r
set.seed(1)
ComputeNormalizedMoranI(scmeta, coords = c("X", "Y"), SE = "SE",
                        CellType = "CellType", nperm = 200, k = 3, ncores = 1)
```

In [ ]:
rrandom.set_seed(1)
moran_z = se.compute_normalized_moran_i(stat_meta, coords=("X", "Y"), se="SE",
                                        cell_type="CellType", nperm=200, k=3,
                                        ncores=1)
print("compute_normalized_moran_i ->", type(moran_z).__name__)
fig, ax = plt.subplots(figsize=(4.6, 3))
moran_z.plot.bar(ax=ax)
ax.set_ylabel("normalised Moran's I (Z)")
ax.set_title("Spatial autocorrelation per SE")
plt.tight_layout(); plt.show()

### 4.E.9 `ComputeMetrics` -> `compute_metrics`

Concordance between true and predicted SE labels: precision / recall / F1 /
F2 per SE within each sample, averaged across the samples where the entry is
defined.

```r
ComputeMetrics(scmeta, SE = "SE", Pred = "cvPred", CellType = NULL,
               Sample = "Sample", metric = "F1")
```

In [ ]:
cv_meta = stat_meta.copy()
cv_meta["cvPred"] = calls.set_index("CID").loc[cv_meta.index, "InitSE"].to_numpy()
f1 = se.compute_metrics(cv_meta, se="SE", pred="cvPred", cell_type=None,
                        sample="Sample", metric="F1")
print("compute_metrics ->", type(f1).__name__, f1.shape)
fig, ax = plt.subplots(figsize=(4.2, 3.6))
im = ax.imshow(f1.to_numpy(), cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(f1.shape[1])); ax.set_xticklabels(f1.columns, rotation=90)
ax.set_yticks(range(f1.shape[0])); ax.set_yticklabels(f1.index)
ax.set_title("ComputeMetrics: F1 (true rows vs predicted cols)")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.show()

### 4.E.10 `buildKNNWeights` -> `build_knn_weights`

A binary k-NN neighbour matrix, optionally radius-capped and optionally
excluding the self-match. Note R uses a **strict** `<` for the radius here,
unlike `.colocalization`'s `<=`.

```r
SpatialEcoTyper:::buildKNNWeights(ref_coords = coords, k = 20, radius = 40,
                                  include.self = TRUE)
```

In [ ]:
knnw = se.build_knn_weights(stat_meta[["X", "Y"]], k=20, radius=40,
                            include_self=True)
knnw_noself = se.build_knn_weights(stat_meta[["X", "Y"]], k=20, radius=40,
                                   include_self=False)
print("build_knn_weights ->", type(knnw).__name__, knnw.shape)
print("  stored neighbours, include_self=True :", knnw.mat.nnz)
print("  stored neighbours, include_self=False:", knnw_noself.mat.nnz)

### 4.E.11 `aggregateByWeights` -> `aggregate_by_weights`

Pushes a per-cell value matrix through a neighbour matrix, optionally
renormalising each output row to sum to 1 and dropping output units backed by
fewer than `min_cells` source cells.

```r
SpatialEcoTyper:::aggregateByWeights(cell2se, weights, sum2one = TRUE,
                                     min.cells = 5)
```

In [ ]:
lvls = sorted(stat_meta["SE"].astype(str).unique())
cell2se = pd.DataFrame(np.eye(len(lvls))[[lvls.index(s) for s in
                                          stat_meta["SE"].astype(str)]],
                       index=stat_meta.index, columns=lvls)
aggw = se.aggregate_by_weights(cell2se, knnw, sum2one=True, min_cells=5)
print("aggregate_by_weights ->", type(aggw).__name__, aggw.shape)
print("  row sums:", float(aggw.sum(axis=1).min()), "..",
      float(aggw.sum(axis=1).max()))
aggw.head().round(4)

### 4.E.12 `ComputeSEAbundanceBySN` -> `compute_se_abundance_by_sn`

SE abundances inside spatial neighbourhoods. With `spot_coords=None` a
regular grid spaced `grid_size` apart is generated and each SN centre is the
component-wise median of its member cells.

```r
seab <- ComputeSEAbundanceBySN(scmeta, spot_coords = NULL, radius = 50,
                               grid.size = 50, SE = "SE", min.cells = 5)
```

In [ ]:
seab = se.compute_se_abundance_by_sn(stat_meta, spot_coords=None, radius=50,
                                     grid_size=50, x="X", y="Y", se="SE",
                                     min_cells=5)
print("compute_se_abundance_by_sn ->", type(seab).__name__, seab.shape)
print("  columns:", list(seab.columns))
payload = [c for c in seab.columns if c not in ("X", "Y")]
fig, axes = plt.subplots(1, len(payload), figsize=(3.1 * len(payload), 3),
                         sharex=True, sharey=True)
for ax, col in zip(np.atleast_1d(axes), payload):
    s = ax.scatter(seab["X"], seab["Y"], c=seab[col], s=8, cmap="magma")
    ax.set_title(col); ax.set_aspect("equal")
    fig.colorbar(s, ax=ax, shrink=0.75)
plt.tight_layout(); plt.show()

### 4.E.13 `SmoothSEAbundances` -> `smooth_se_abundances`

Averages spot-level SE abundances over the `k` nearest spots. The default
`k = 7` matches a Visium hex grid (self plus 6 immediate neighbours) when
`include_self = True`.

```r
SmoothSEAbundances(as.matrix(seab[, -c(1, 2)]), seab[, c("X", "Y")], k = 7,
                   include.self = TRUE, min.neighbors = 3)
```

In [ ]:
smooth = se.smooth_se_abundances(seab[payload], seab[["X", "Y"]], k=7,
                                 x="X", y="Y", include_self=True,
                                 min_neighbors=3)
print("smooth_se_abundances ->", type(smooth).__name__, smooth.shape)
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(seab[payload[0]].to_numpy(), label="raw")
ax.plot(smooth.loc[seab.index, payload[0]].to_numpy(), label="smoothed (k=7)")
ax.set_xlabel("spatial neighbourhood"); ax.set_ylabel(payload[0])
ax.set_title("SmoothSEAbundances"); ax.legend()
plt.tight_layout(); plt.show()

### 4.E.14 `AverageMarkerExpression` -> `average_marker_expression`

Pseudo-bulks the expression by SE, averages each gene set, log2(x+1)s and
z-scores across SEs. Upstream defaults to the SE consensus markers shipped in
the R package's `inst/extdata`; those `.rds` files are not redistributable,
so `genesets` is a required argument here.

```r
AverageMarkerExpression(obj, group.by = "SE", genesets = genesets)
```

In [ ]:
panel = {
    "MHC-II": [g for g in ("HLA-DRA", "HLA-DRB1", "CD74") if g in genes],
    "T cell": [g for g in ("CD3D", "CD3E", "CD2") if g in genes],
    "Myeloid": [g for g in ("CD68", "CD14", "ITGAX") if g in genes],
}
panel = {k: v for k, v in panel.items() if v}
print("gene sets:", {k: len(v) for k, v in panel.items()})
avg_marker, marker_fig = se.average_marker_expression(
    sub_norm, result.metadata["SE"].to_numpy(), gene_names=genes,
    genesets=panel, plot=True)
print("average_marker_expression -> tuple(DataFrame, Figure)")
print("  avg_exprs:", avg_marker.shape)
avg_marker.round(3)

### 4.F Visualisation

### 4.F.1 `SpatialView` -> `spatial_view`

The spatial scatter behind every figure in the paper: colour cells or spots
by a metadata column or a gene, with optional highlight / control sets.

```r
SpatialView(scmeta, by = "SE", pt.size = 0.5) + ggtitle("Spatial ecotypes")
```

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
se.spatial_view(result.metadata, by="SE", x="X", y="Y", pt_size=1.4,
                coord_fix=True, ax=axes[0])
axes[0].set_title("SpatialView(by = 'SE')")
se.spatial_view(result.metadata, by="CellType", x="X", y="Y", pt_size=1.4,
                coord_fix=True, ax=axes[1])
axes[1].set_title("SpatialView(by = 'CellType')")
plt.tight_layout(); plt.show()

### 4.F.2 `HeatmapView` -> `heatmap_view`

A ComplexHeatmap-shaped matplotlib heatmap: three-stop colour ramp, optional
row / column dendrograms, splits, and up to four annotation tracks.

```r
HeatmapView(mat, breaks = c(0, 0.6, 1.2),
            colors = c("#ffffd9", "#edf8b1", "#225ea8"))
```

In [ ]:
hm_mat = avg_marker.copy()
hm_fig = se.heatmap_view(hm_mat, breaks=(-1.5, 0, 1.5),
                         colors=("#2166ac", "#f7f7f7", "#b2182b"),
                         cluster_rows=False, cluster_cols=False,
                         name="hmap", legend_title="z-score",
                         figsize=(5.4, 2.6))
print("heatmap_view ->", type(hm_fig).__name__)
plt.show()

### 4.F.3 `drawRectangleAnnotation` -> `draw_rectangle_annotation`

Outlines the blocks where a row annotation and a column annotation agree -
in R it decorates a drawn `HeatmapList`; here it decorates the returned
`Figure`.

```r
ht <- HeatmapView(mat, ...)
drawRectangleAnnotation(ht, rows = rowann, columns = colann)
```

In [ ]:
sq = coassoc.iloc[:24, :24]
sq_rows = [str(v).split("_")[0] for v in sq.index]
sq_cols = [str(v).split("_")[0] for v in sq.columns]
ht = se.heatmap_view(sq, breaks=(-1, 0, 1),
                     colors=("#2166ac", "#f7f7f7", "#b2182b"),
                     cluster_rows=False, cluster_cols=False, name="hmap",
                     show_row_names=False, show_column_names=False,
                     figsize=(5.0, 4.6))
ht = se.draw_rectangle_annotation(ht, rows=sq_rows, columns=sq_cols,
                                  col="black", heatmap_name="hmap")
print("draw_rectangle_annotation ->", type(ht).__name__)
plt.show()

### 4.F.4 `CooccurrenceHeatmapView` -> `cooccurrence_heatmap_view`

The colocalization heatmap: a square cell-state matrix with SE and cell-type
row annotations and, optionally, significance stars from the p-values.
Default breaks are the 55 / 75 / 90 % quantiles of the matrix.

```r
CooccurrenceHeatmapView(coloc$ColocIndex, pval = coloc$Pval)
```

In [ ]:
co_fig = se.cooccurrence_heatmap_view(coloc["ColocIndex"], pval=coloc["Pval"],
                                      cluster_rows=False, cluster_cols=False,
                                      figsize=(6.4, 5.4))
print("cooccurrence_heatmap_view ->", type(co_fig).__name__)
plt.show()

## 5. The class API mirror

Everything in section 4.B can be written as a single method chain. The class
keeps the staged intermediates so you can inspect the fused graph, the
neighbourhood metadata or the PCA without re-running.

In [ ]:
obj = se.SpatialEcoTyper(normdata, meta, gene_names=genes,
                         radius=50, resolution=0.5, nfeatures=300,
                         ncores=1, verbose=False)
print("before .run():", obj)
obj = obj.run()
print("after  .run():", obj)

`SpatialEcoTyperResult` is what R returns as `list(obj = <Seurat>,
metadata = <data.frame>)`. Because there is no Seurat object in Python, its
contents are exposed directly - the mapping onto AnnData slots is:

| R | `SpatialEcoTyperResult` | AnnData analogue |
|---|---|---|
| `obj$SE` | `.spot_metadata["SE"]` | `adata.obs["SE"]` (neighbourhood-level) |
| `res$metadata$SE` | `.metadata["SE"]` | `adata.obs["SE"]` (cell-level) |
| `obj[["RNA"]]$counts` | `.fused` | `adata.obsm["fused"]` / `.obsp` |
| `Embeddings(obj, "pca")` | `.pca` | `adata.obsm["X_pca"]` |
| `obj@graphs$RNA_snn` | `.snn` | `adata.obsp["connectivities"]` |
| `obj$seurat_clusters` | `.clusters` | `adata.obs["leiden"]`-style codes |
| `obj@project.name` | `.project_name` | `adata.uns["spatialecotyper"]` |

In [ ]:
r = obj.result
print("spot_metadata:", r.spot_metadata.shape)
print("metadata     :", r.metadata.shape)
print("fused        :", r.fused.shape, type(r.fused).__name__)
print("pca          :", r.pca.shape)
print("snn          :", r.snn.shape, type(r.snn).__name__)
print("clusters     :", np.unique(r.clusters))
print("project_name :", r.project_name)
print("obj.se (per-cell SE Series):")
print(obj.se.value_counts().sort_index())
print("identical to the functional API run above:",
      bool((obj.se.to_numpy() == result.metadata["SE"].to_numpy()).all()))

For an AnnData-native workflow, the results drop straight in:

```python
adata.obs["SE"] = obj.se.reindex(adata.obs_names)
adata.uns["spatialecotyper"] = {"project_name": obj.result.project_name}
adata.obsm["X_se_pca"] = obj.result.pca
```

## 6. Common pitfalls

### 6.1 Genes in rows, cells in columns

Every algorithmic function in this package expects **genes x cells**, the R /
Seurat convention - the opposite of `scanpy`'s cells x genes `AnnData.X`.
Feeding a transposed matrix does not raise; it silently produces nonsense.
The one exception is `NMFpredict`, which *returns* cells x SE because R
transposes `H` on the way out. Check with a shape assertion.

In [ ]:
print("normdata is genes x cells:", normdata.shape,
      "== (len(genes), len(cells)) ==", (len(genes), len(cells)))
print("nmf_predict returns cells x SE:", H.shape,
      "-> rows are cells, columns are", H_ses)
print("from AnnData you would pass adata.X.T, not adata.X")

### 6.2 R is base-1, Python is base-0

Anything that surfaces an index differs by one. `nmf_clustering` returns
1-based cluster labels because `NMF::predict` does; `find_clusters` returns
0-based Louvain communities because Seurat's `seurat_clusters` factor is
0-based, and `spatial_ecotyper` then writes `NewSE{cluster + 1}`. Read the
`+1` in the source before comparing label vectors.

In [ ]:
print("nmf_clustering labels (1-based, like NMF::predict):",
      np.unique(clust["labels"]))
print("spatial_ecotyper SE labels (cluster + 1):",
      sorted(set(result.spot_metadata['SE'])))
print("R's `nn2` returns 1-based neighbour indices; pyspatialecotyper's "
      "internal `_nn2` returns 0-based and marks padding with -1")

### 6.3 R's RNG stream, and `pyspatialecotyper.rrandom`

The permutation tests, the NMF restart seeds and the down-sampling all draw
from R's Mersenne-Twister. `set.seed(1)` in R is `rrandom.set_seed(1)` here -
**not** `np.random.seed(1)`, which gives a different stream and therefore
different p-values. `rrandom` reproduces `unif_rand` bit-identically, the
post-R-3.6 rejection sampler behind `sample()` identically, and the inversion
`rnorm` to 2.2e-16.

In [ ]:
rrandom.set_seed(42)
print("rrandom.runif(5) after set_seed(42):", rrandom.runif(5).round(8))
print("(bit-compared against R's own stream in "
      "tests/test_exact_match.py::test_r_rng_matches_r)")
rrandom.set_seed(1)
p_a = se.coassociation_test(coassoc, nperm=200)
rrandom.set_seed(1)
p_b = se.coassociation_test(coassoc, nperm=200)
print("same seed reproduces the same p-values:",
      bool((p_a.to_numpy() == p_b.to_numpy()).all()))

### 6.4 The `..` separator in metacell column names

`get_spatial_metacells` names its columns `"<SpotID>..<CellType>"`, and every
downstream step splits on the **last** `..`. Cell types or spot IDs
containing a literal `..` will break the split. Note also that
`integrate_spatial_ecotyper` uses the *same* separator one level up, for
`"<Sample>..<InitSE>"`.

In [ ]:
print("metacell column example :", ncem_cols[0])
print("  spot :", ncem_cols[0].rsplit("..", 1)[0])
print("  type :", ncem_cols[0].rsplit("..", 1)[1])
print("integration column example:", multi["cluster_SE"].index[0],
      "->", multi["cluster_SE"].index[0].split("..")[0])

### 6.5 `normalization_method="SCT"` raises `NotImplementedError`

Upstream's `IntegrateSpatialEcoTyper` accepts `normalization.method = "SCT"`,
which would require a port of `SCTransform`. That is out of scope, and the
port refuses loudly rather than silently falling back to `"None"`.

In [ ]:
try:
    se.integrate_spatial_ecotyper(result_list={"SKCM": res_skcm},
                                  data_list={"SKCM": normdata},
                                  gene_names=genes,
                                  normalization_method="SCT")
except NotImplementedError as exc:
    print("NotImplementedError:", exc)

### 6.6 The bundled MERSCOPE recovery models are not redistributed

`RecoverSE` and `DeconvoluteSE` in R default to `.rds` models shipped in the
package's `inst/extdata`, including the published `SE01..SE11 -> SE1..SE9`
relabelling. Those files are covered by the Stanford non-commercial licence
and are not redistributed here, so `Ws` / `W` must be supplied explicitly -
train them with `nmf_generate_w_list` as in section 4.D.2. Likewise,
`AverageMarkerExpression` requires `genesets`.

In [ ]:
try:
    se.recover_se(sub_norm, genes, list(result.metadata.index),
                  result.metadata["CellType"].to_numpy(), Ws={})
except (ValueError, TypeError, StopIteration) as exc:
    print(type(exc).__name__, ":", exc)
try:
    se.average_marker_expression(sub_norm, result.metadata["SE"].to_numpy(),
                                 gene_names=genes, genesets=None, plot=False)
except (ValueError, NotImplementedError) as exc:
    print(type(exc).__name__, ":", exc)

### 6.7 `getSN` and tied nearest neighbours

`README.md` documents the one pre-registered gate the port does not clear:
when two spatial neighbourhoods have *exactly identical* cell-type-specific
metacell profiles, their distances tie at the k-th neighbour boundary and
`RANN`'s ANN kd-tree and `scipy.spatial.cKDTree` break the tie differently.
Both answers are correct k-NN; the tie is inherent. Below is how many
neighbourhood pairs in this crop are exactly tied for one cell type.

In [ ]:
emb_ct, names_ct = emb_list[ct0]
pts = np.ascontiguousarray(emb_ct[:20].T)
uniq = np.unique(pts.round(12), axis=0)
print(f"cell type {ct0}: {pts.shape[0]} neighbourhoods, "
      f"{uniq.shape[0]} distinct 20-PC profiles, "
      f"{pts.shape[0] - uniq.shape[0]} exact duplicates")

## 7. Where to go next

* [`README.md`](../README.md) - the public API and the measured parity table
  (every number reproducible with `pytest tests/test_exact_match.py`).
* [`AUDIT.md`](../AUDIT.md) - the R-function coverage table, 35/35 exported
  functions.
* [`MATH.md`](../MATH.md) - the derivations behind the one bounded-epsilon
  acceleration rewrite.
* [`ITERATION_LOG.md`](../ITERATION_LOG.md) and
  [`examples/evolution.ipynb`](evolution.ipynb) - the per-iteration record.
* [`examples/compare_R_vs_Python.ipynb`](compare_R_vs_Python.ipynb) -
  pipeline-level parity against R, one visualisation per manifest output.
* [`examples/function_by_function_R_parity.ipynb`](function_by_function_R_parity.ipynb) -
  the R-to-Python migration dictionary: every R parameter in a table, with a
  numerical comparison per function.
* Upstream R package:
  <https://github.com/digitalcytometry/spatialecotyper>
* Paper: Zhang *et al.*, *Nature* (2026),
  [doi:10.1038/s41586-026-10452-4](https://doi.org/10.1038/s41586-026-10452-4).

Licence note: Spatial EcoTyper is distributed under the **Stanford
Non-Commercial Software License Agreement** (docket S24-045) and this port is
a derivative work that inherits that restriction. It is *not* MIT-licensed.